In [27]:
"""Upload a PDF through the same Preview pipeline the frontend uses.

Flow: login → POST /api/upload → poll GET /api/pdf/jobs/{job_id} → GET /api/pdf/jobs/{job_id}/result
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import requests

BASE = "http://localhost:8000"
PDF_PATH = Path("/Users/alokhegde/Dhara/media_to_upload1762321618.pdf")
EMAIL = "alokhegde2221@gmail.com"
PASSWORD = "12345678"

assert PDF_PATH.is_file(), f"Missing PDF: {PDF_PATH}"
print(f"PDF: {PDF_PATH.name} ({PDF_PATH.stat().st_size:,} bytes)")

# Login (or signup if the user does not exist yet)
r = requests.post(f"{BASE}/api/login", json={"email": EMAIL, "password": PASSWORD})
if r.status_code == 401:
    r = requests.post(
        f"{BASE}/api/signup",
        json={"email": EMAIL, "password": PASSWORD, "name": "Notebook"},
    )
r.raise_for_status()
auth = r.json()
TOKEN = auth["token"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"Logged in as {auth['email']} (role={auth.get('role')})")

PDF: media_to_upload1762321618.pdf (3,479,370 bytes)
Logged in as alokhegde2221@gmail.com (role=data_steward)


In [28]:
# Same endpoint the console uses after file pick: POST /api/upload (type=pdf)
with PDF_PATH.open("rb") as f:
    up = requests.post(
        f"{BASE}/api/upload",
        headers=HEADERS,
        data={"type": "pdf"},
        files={"file": (PDF_PATH.name, f, "application/pdf")},
    )
print("POST /api/upload", up.status_code)
up.raise_for_status()
upload = up.json()
JOB_ID = upload["job_id"]
print(json.dumps(upload, indent=2))
print(f"\njob_id = {JOB_ID}")

POST /api/upload 200
{
  "job_id": "789434515c3641b8914bffb6bfc28956",
  "source_type": "pdf"
}

job_id = 789434515c3641b8914bffb6bfc28956


In [29]:
# Poll until extraction finishes (same as Processing → Preview unlock)
STATUS = None
for i in range(600):
    st = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}", headers=HEADERS)
    st.raise_for_status()
    job = st.json()
    STATUS = job.get("status")
    pct = job.get("percent") or job.get("progress", {}).get("percent")
    msg = job.get("message") or (job.get("progress") or {}).get("message") or ""
    if i % 5 == 0 or STATUS not in ("queued", "running", "processing"):
        print(f"[{i:03d}] status={STATUS} percent={pct} {msg}")
    if STATUS in ("done", "completed_with_errors", "error", "failed"):
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Job {JOB_ID} still {STATUS} after polling")

if STATUS not in ("done", "completed_with_errors"):
    raise RuntimeError(f"Job failed: {json.dumps(job, indent=2)[:2000]}")
print("\nJob finished:")
print(json.dumps({k: job.get(k) for k in ("job_id", "status", "filename", "percent", "message", "source_type")}, indent=2))

[000] status=done percent=100 Complete

Job finished:
{
  "job_id": "789434515c3641b8914bffb6bfc28956",
  "status": "done",
  "filename": "media_to_upload1762321618.pdf",
  "percent": 100,
  "message": "Complete",
  "source_type": "pdf"
}


In [30]:
# Preview stage payload — same as PdfReview: GET /api/pdf/jobs/{job_id}/result
res = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}/result", headers=HEADERS)
print("GET /api/pdf/jobs/.../result", res.status_code)
res.raise_for_status()
preview = res.json()
TABLES = preview.get("tables") or []

print(f"filename: {preview.get('filename')}")
print(f"is_batch: {preview.get('is_batch')}")
print(f"table count: {len(TABLES)}")
print()

# Compact card list (what the Preview UI shows in the left/list view)
rows = []
for t in TABLES:
    rows.append({
        "table_id": t.get("table_id"),
        "page": t.get("page"),
        "title": (t.get("title") or "")[:80],
        "n_cols": len(t.get("columns") or []),
        "n_rows": len(t.get("rows") or []),
        "human_review_needed": bool(t.get("human_review_needed")),
        "human_review_reason": t.get("human_review_reason"),
        "source_location": t.get("source_location"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    print(json.dumps(rows, indent=2)[:8000])

GET /api/pdf/jobs/.../result 200
filename: media_to_upload1762321618.pdf
is_batch: False
table count: 162



,table_id,page,title,n_cols,n_rows,human_review_needed,human_review_reason,source_location
0,DDI_DES_DEL_VS_03609,3,Team Associated in the Preparation of Publication,2,5,False,None,Page 3
1,DDI_DES_DEL_VS_03610,7,"Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal...",4,3,False,None,Page 7
2,DDI_DES_DEL_VS_03611,8,"Classification of Commercial, Fibre, and Horti...",4,3,True,title_filled_by_ai,Page 8
3,DDI_DES_DEL_VS_03612,9,"Area : In lakh hectares, Production: In lakh T...",8,19,False,None,Page 9
4,DDI_DES_DEL_VS_03613,10,"Area : In lakh hectares, Production: In lakh T...",8,25,False,None,Page 10
...,...,...,...,...,...,...,...,...
157,DDI_DES_DEL_VS_03766,163,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",8,32,False,None,Page 163
158,DDI_DES_DEL_VS_03767,164,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",17,32,False,None,Page 164
159,DDI_DES_DEL_VS_03768,165,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",11,32,False,None,Page 165
160,DDI_DES_DEL_VS_03769,166,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",8,32,False,None,Page 166


In [5]:
# Inspect one Preview table in full (columns + first rows) — pick by index
IDX = 0
t = TABLES[IDX]
print(f"table_id: {t.get('table_id')}")
print(f"title:    {t.get('title')}")
print(f"page:     {t.get('page')}")
print(f"review:   needed={t.get('human_review_needed')} reason={t.get('human_review_reason')}")
print()
cols = t.get("columns") or []
if cols and isinstance(cols[0], dict):
    col_names = [c.get("name") for c in cols]
else:
    col_names = list(cols)
print("columns:", col_names)
print(f"rows ({len(t.get('rows') or [])} total), first 8:")
for row in (t.get("rows") or [])[:8]:
    print(row)

# Full JSON for this table (can be large)
# print(json.dumps(t, indent=2, default=str)[:5000])

table_id: DDI_DES_DEL_VS_03447
title:    Team Associated in the Preparation of Publication
page:     3
review:   needed=False reason=None

columns: ['Sri K. Gururaja Rao', 'Additional Director']
rows (5 total), first 8:
['Sri. Shashidhara.T', 'Joint Director']
['Smt. Girija.T.S', 'Deputy Director']
['Smt. Ashwini kumari .H.B', 'Assistant Director']
['Smt. Mamatha .T.K', 'Assistant Statistical Officer']
['Kum.Nethravathi.M', 'Enumerator']


In [31]:
# Optional: skip re-upload — load Preview for an already-finished job_id
# (this PDF already ran as 7e77554c3af64c71809a0363e151304e → 162 tables)
JOB_ID = "789434515c3641b8914bffb6bfc28956"

res = requests.get(f"{BASE}/api/pdf/jobs/{JOB_ID}/result", headers=HEADERS)
res.raise_for_status()
preview = res.json()
TABLES = preview.get("tables") or []
print(f"job_id={JOB_ID}  filename={preview.get('filename')}  tables={len(TABLES)}")

# How many need human review (Preview badges)
n_review = sum(1 for t in TABLES if t.get("human_review_needed"))
print(f"human_review_needed: {n_review} / {len(TABLES)}")

# Persist a slim summary next to the notebook for inspection
summary_path = Path("/Users/alokhegde/Dhara/chat_preview_summary.json")
if summary_path.is_file():
    print(f"\nAlso on disk: {summary_path} ({summary_path.stat().st_size:,} bytes)")
    slim = json.loads(summary_path.read_text())
    print(f"cached summary table_count={slim.get('table_count')}")

job_id=789434515c3641b8914bffb6bfc28956  filename=media_to_upload1762321618.pdf  tables=162
human_review_needed: 1 / 162

Also on disk: /Users/alokhegde/Dhara/chat_preview_summary.json (186,948 bytes)
cached summary table_count=162


# Chat interface — llm_preview catalog

Send a **compact** table stub (no rows/columns) to `POST /api/llm_preview/jobs/{job_id}/llm_preview_data`, then read it back with GET.

Default POST stores `{table_id, name, summary}` from the stubs — **no LangChain**. Pass `"force_summarize": true` only if you want an LLM rewrite (needs API key).

In [32]:
# Compact stubs only — LLM does not need full Preview rows/columns
def compact_tables_for_llm(tables):
    out = []
    for t in tables:
        if not t.get("table_id"):
            continue
        entry = {
            "table_id": t.get("table_id"),
            "page": t.get("page"),
            "title": t.get("title"),
            "banner_table_id": t.get("banner_table_id"),
            "short_description": t.get("short_description") or t.get("description") or t.get("summary"),
            "human_review_needed": t.get("human_review_needed"),
            "human_review_reason": t.get("human_review_reason"),
        }
        # drop empty/None values to keep the payload small
        out.append({k: v for k, v in entry.items() if v is not None and v != ""})
    return out

CHAT_TABLES = compact_tables_for_llm(TABLES)
payload_bytes = len(json.dumps(CHAT_TABLES).encode("utf-8"))
full_bytes = len(json.dumps(TABLES, default=str).encode("utf-8"))
print(f"tables: {len(CHAT_TABLES)}")
print(f"compact payload: {payload_bytes:,} bytes  (full Preview tables would be ~{full_bytes:,} bytes)")
print("sample stub:")
print(json.dumps(CHAT_TABLES[0] if CHAT_TABLES else {}, indent=2))

# POST → stores catalog from stubs (no LangChain). Use force_summarize=True only if you want LLM rewrite.
post = requests.post(
    f"{BASE}/api/llm_preview/jobs/{JOB_ID}/llm_preview_data",
    headers={**HEADERS, "Content-Type": "application/json"},
    json={"tables": CHAT_TABLES},
    timeout=120,
)
print("\nPOST /api/llm_preview/jobs/.../llm_preview_data", post.status_code)
if not post.ok:
    print(post.text[:2000])
post.raise_for_status()
post_body = post.json()
print(f"returned catalog entries: {len(post_body.get('tables') or [])}")
print(f"source_table_count: {post_body.get('source_table_count')}")
print(f"updated_at: {post_body.get('updated_at')}")
print("\nfirst 5 catalog rows from POST response:")
print(json.dumps((post_body.get("tables") or [])[:5], indent=2))

tables: 162
compact payload: 30,941 bytes  (full Preview tables would be ~1,089,441 bytes)
sample stub:
{
  "table_id": "DDI_DES_DEL_VS_03609",
  "page": 3,
  "title": "Team Associated in the Preparation of Publication",
  "human_review_needed": false
}

POST /api/llm_preview/jobs/.../llm_preview_data 200
returned catalog entries: 162
source_table_count: 162
updated_at: 1792402020.1117854

first 5 catalog rows from POST response:
[
  {
    "table_id": "DDI_DES_DEL_VS_03609",
    "name": "Team Associated in the Preparation of Publication",
    "summary": "Team Associated in the Preparation of Publication"
  },
  {
    "table_id": "DDI_DES_DEL_VS_03610",
    "name": "Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated",
    "summary": "Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated"
  },
  {
    "table_id": "DDI_DES_DEL_VS_03611",
    "nam

In [35]:
# GET stored catalog (same job_id — what chat mode reads back)
get = requests.get(
    f"{BASE}/api/llm_preview/jobs/{JOB_ID}/llm_preview_data",
    headers=HEADERS,
    timeout=60,
)
print("GET /api/llm_preview/jobs/.../llm_preview_data", get.status_code)
if not get.ok:
    print(get.text[:2000])
get.raise_for_status()
llm_preview = get.json()
print("llm_preview",llm_preview)
CATALOG = llm_preview.get("tables") or []

print(f"job_id: {llm_preview.get('job_id')}")
print(f"catalog tables: {len(CATALOG)}")
print(f"pipeline_step: {llm_preview.get('pipeline_step')}")
print(f"last_turns: {len(llm_preview.get('last_turns') or [])}")
print()

try:
    import pandas as pd
    display(pd.DataFrame(CATALOG))
except Exception:
    print(json.dumps(CATALOG[:20], indent=2))

# Persist for inspection next to the notebook
out_path = Path("/Users/alokhegde/Dhara/chat_llm_preview_data.json")
out_path.write_text(json.dumps(llm_preview, indent=2, default=str))
print(f"\nwrote {out_path}")

GET /api/llm_preview/jobs/.../llm_preview_data 200
llm_preview {'job_id': '789434515c3641b8914bffb6bfc28956', 'tables': [{'name': 'Team Associated in the Preparation of Publication', 'summary': 'Team Associated in the Preparation of Publication', 'table_id': 'DDI_DES_DEL_VS_03609'}, {'name': 'Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated', 'summary': 'Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengalgram, Groundnut, Sunflower, Safflower and Cotton crops are given for irrigated', 'table_id': 'DDI_DES_DEL_VS_03610'}, {'name': 'Classification of Commercial, Fibre, and Horticultural Crops', 'summary': 'Classification of Commercial, Fibre, and Horticultural Crops', 'table_id': 'DDI_DES_DEL_VS_03611'}, {'name': 'Area : In lakh hectares, Production: In lakh Tonnes, Yield : In Kgs per hectare', 'summary': 'Area : In lakh hectares, Production: In lakh Tonnes, Yield : In Kgs per hectare', 'table_id': 'DDI_DES_DEL_VS_0361

,name,summary,table_id
0,Team Associated in the Preparation of Publication,Team Associated in the Preparation of Publication,DDI_DES_DEL_VS_03609
1,"Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal...","Rice, Jowar, Bajra, Maize, Ragi, Wheat, Bengal...",DDI_DES_DEL_VS_03610
2,"Classification of Commercial, Fibre, and Horti...","Classification of Commercial, Fibre, and Horti...",DDI_DES_DEL_VS_03611
3,"Area : In lakh hectares, Production: In lakh T...","Area : In lakh hectares, Production: In lakh T...",DDI_DES_DEL_VS_03612
4,"Area : In lakh hectares, Production: In lakh T...","Area : In lakh hectares, Production: In lakh T...",DDI_DES_DEL_VS_03613
...,...,...,...
157,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",DDI_DES_DEL_VS_03766
158,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",DDI_DES_DEL_VS_03767
159,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",DDI_DES_DEL_VS_03768
160,"FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...","FULLY REVISED ESTIMATES OF AREA, PRODUCTION AN...",DDI_DES_DEL_VS_03769



wrote /Users/alokhegde/Dhara/chat_llm_preview_data.json


# Chat over Preview — backend `llm_input`

Send a natural-language **user_request** to the API. The server runs LangChain intent + action dispatch on the same `job_id` as guided Preview (catalog auto-builds if missing).

Optional follow-up: `execute_preview_action` for confirmed delete or writing download files to disk.

Example: *download tables related to condiment and spices*.

In [36]:
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

BACKEND = Path("/Users/alokhegde/Dhara/backend").resolve()
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

# Import client directly (avoids pulling optional DB drivers into Jupyter)
from app.services.llm_preview.client import (
    execute_preview_action,
    get_llm_preview_data,
    post_llm_input,
    post_llm_preview_data,
)
from app.services.llm_preview.schemas import LlmInputResponse


def query_preview_catalog(
    user_request: str,
    *,
    job_id: Optional[str] = None,
    headers: Optional[dict] = None,
    base: str = BASE,
    tables: Optional[List[dict]] = None,
    **kwargs,
) -> Dict[str, Any]:
    """Calls POST .../llm_input (LangChain + action execution on server)."""
    for k in ("catalog", "api_key", "model"):
        kwargs.pop(k, None)
    return post_llm_input(
        base_url=base,
        job_id=job_id or JOB_ID,
        user_request=user_request,
        headers=headers or HEADERS,
        tables=tables,
    )


print("Backend llm_preview client ready")


ModuleNotFoundError: No module named 'psycopg2'

In [23]:
# Example: steward query → POST llm_input (LangChain + table_data on server)
USER_PROMPT = "download tables related to condiment and spices"

result = query_preview_catalog(
    USER_PROMPT,
    tables=TABLES,  # optional Preview overlay; catalog auto-builds if needed
    job_id=JOB_ID,
)

print("type:", result["type"])
print("rationale:", result["rationale"])
print("executed:", result.get("executed"))
print(f"matches: {result.get('count', len(result.get('table_ids') or []))}")
print()
print("table_names (id → name):")
print(json.dumps(result["table_names"], indent=2, ensure_ascii=False))


type: bulk_download
rationale: These tables specifically relate to condiment and spice crops.
matches: 3

table_names (id → name):
{
  "DDI_DES_DEL_VS_03605": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICE CROPS IN KARNATAKA FOR THE YEAR 2023-24",
  "DDI_DES_DEL_VS_03606": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICECROPS IN KARNATAKA FOR THE YEAR 2023-24",
  "DDI_DES_DEL_VS_03607": "FULLY REVISED ESTIMATES OF AREA, PRODUCTION AND YIELD OF CONDIMENT AND SPICECROPS IN KARNATAKA"
}


/opt/homebrew/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=CatalogAction(type='bulk_...iment and spice crops.'), input_type=CatalogAction])
  return self.__pydantic_serializer__.to_python(


## Optional follow-up

`execute_preview_action` (backend client) confirms **delete** or writes **download** files to disk. Find/review responses already include `table_data` from `llm_input`.


In [38]:
from pathlib import Path

# Re-import so kernel never uses a stale local execute_preview_action
from app.services.llm_preview.client import execute_preview_action

DOWNLOAD_DIR = Path("/Users/alokhegde/Dhara")
print("execute_preview_action:", execute_preview_action)


Using backend execute_preview_action for delete confirm / disk download


In [39]:
# Find/bulk_find: table_data is already in `result` from llm_input
executed = execute_preview_action(
    result,
    base_url=BASE,
    job_id=JOB_ID,
    headers=HEADERS,
    download_dir=DOWNLOAD_DIR,
)

print("type:", executed["type"])
print("executed:", executed.get("executed"))
print("count:", executed.get("count"))
print("table_ids:", executed.get("table_ids"))
if executed.get("path"):
    print("path:", executed["path"])

if result.get("table_data"):
    tid0 = result["table_ids"][0]
    t0 = result["table_data"][tid0]
    print()
    print(f"sample {tid0}:")
    print("  title:", t0.get("title"))
    print("  page:", t0.get("page"))
    print("  n_cols:", len(t0.get("columns") or []))
    print("  n_rows:", len(t0.get("rows") or []))

# Delete (safe by default):
# execute_preview_action(result, base_url=BASE, job_id=JOB_ID, headers=HEADERS, confirm_delete=True)


TypeError: execute_preview_action() got an unexpected keyword argument 'base_url'